# Unsway — Phase 3 on Colab

This notebook contains no research logic. It installs the repository and calls the versioned Phase 1–3 CLI entry points.

Before running: select **Runtime → Change runtime type → GPU**.

In [ ]:
!nvidia-smi
import torch

assert torch.cuda.is_available(), "Enable a GPU runtime before continuing"
print("CUDA device:", torch.cuda.get_device_name(0))

## Clone and install

The printed Git revision is part of the run provenance.

In [ ]:
import os
from pathlib import Path

repo = Path("/content/Unsway")
if not repo.exists():
    !git clone https://github.com/idris404/Unsway.git /content/Unsway
%cd /content/Unsway
!git pull --ff-only
!git rev-parse HEAD
!pip -q install uv
!uv sync --extra dev

## Persistent backup location

Compute stays on Colab's fast local disk. Artifacts and reports are copied to Drive after every expensive stage.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
drive_dir = Path("/content/drive/MyDrive/Unsway/phase3")
drive_dir.mkdir(parents=True, exist_ok=True)
os.environ["UNSWAY_DRIVE_DIR"] = str(drive_dir)
print("Backups:", drive_dir)

## Rebuild ignored Phase 1 and Phase 2 artifacts

Both commands verify the pinned source and dataset checksums.

In [ ]:
!uv run unsway-phase1 --config configs/phase1.yaml
!uv run unsway-phase2 --config configs/phase2.yaml

## Extract the full activation corpus

This reads train and validation only. Test examples remain untouched.

In [ ]:
!uv run unsway-phase3 --config configs/phase3.yaml --stage extract
!mkdir -p "$UNSWAY_DRIVE_DIR/data" "$UNSWAY_DRIVE_DIR/reports"
!rsync -a data/processed/phase3/ "$UNSWAY_DRIVE_DIR/data/"
!cp reports/phase3_extraction.json "$UNSWAY_DRIVE_DIR/reports/"

## Train the full Top-K SAE

The production configuration trains a 768 → 6,144 SAE for 15 epochs. Monitor validation explained variance and dead features in the logs.

In [ ]:
!uv run unsway-phase3 --config configs/phase3.yaml --stage train
!cp data/processed/phase3/sae.safetensors "$UNSWAY_DRIVE_DIR/data/"
!cp reports/phase3_training.json "$UNSWAY_DRIVE_DIR/reports/"

## Rank on train and confirm on validation

In [ ]:
!uv run unsway-phase3 --config configs/phase3.yaml --stage analyze
!cp reports/phase3_features.json "$UNSWAY_DRIVE_DIR/reports/"

## Inspect the final diagnostics

Do not select a feature from train AUROC alone. Validation oriented AUROC and SAE reconstruction quality are the gates for Phase 4.

In [ ]:
import json
from pprint import pprint

training = json.loads(Path("reports/phase3_training.json").read_text())
features = json.loads(Path("reports/phase3_features.json").read_text())
print("Final SAE validation metrics:")
pprint(training["history"][-1]["validation"])
print("\nTop feature:")
pprint(features["top_features"][0])
print("\nRaw-neuron baseline:")
pprint(features["raw_neuron_baseline"])

## Optional restore after a disconnected runtime

After cloning and mounting Drive again, run this cell before restarting at `train` or `analyze`.

In [ ]:
!mkdir -p data/processed/phase3 reports
!rsync -a "$UNSWAY_DRIVE_DIR/data/" data/processed/phase3/
!rsync -a "$UNSWAY_DRIVE_DIR/reports/" reports/